In [26]:
import pandas as pd
import numpy as np
import geopandas as gpd
from sklearn.ensemble import RandomForestRegressor
from utils.data_utils import evaluate_rf
import utils.cross_validation as cval
import random 
from shapely.geometry import box

import threading


In [27]:
import utils.analysis_functions as af

In [28]:
TEST_SIZE = 0.4
RANDOM_KEY = 21
BATCH_SIZE=16
diversity_type = "functional" #"functional " "species" "combined"

biome_mapping = {
    0: 'Boreal and Tundra forests',
    1: 'Flooded grasslands',
    2: 'Mangroves',
    3: 'Mediterranean woodlands',
    4: 'Temperate broadleaf forests',
    5: 'Temperate conifer forests',
    6: 'Temperate grasslands',
    7: 'Tropical',
    8: 'Tropical',
    9: 'Tropical',
    10: 'Tropical',
    11: 'Boreal and Tundra forests',
    12: 'Xeric shrublands',
    13: np.nan
}

if diversity_type == "combined":
        df_name = "data/final/combined_df_gc.csv"
        div_type = "comb"
elif diversity_type == "functional":
    df_name = "data/final/fd_df_gc.csv"
    div_type = "f"
else:
    df_name = "data/final/sd_df_gc.csv"
    div_type = "s"

df= pd.read_csv(df_name)

ecoregions=cval.process_ecoregion("data/Ecoregions/Ecoregions2017.shp")

ecoregions=ecoregions[['ECO_NAME', 'geometry']]

#Preprocessing the data for Random forest regression

df = df[~df['biome'].isin([13, 1, 2, 3])]

df["biome"] = df["biome"].replace(biome_mapping)

df=cval.assign_spatial_groups(df, grid_size=1.0)

gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["lon"], df["lat"]),
    crs="EPSG:4326"  # WGS84
)

joined = gpd.sjoin(gdf, ecoregions, how="left", predicate="within")

biome_dfs = {k: v for k, v in joined.groupby('biome')}


In [69]:
test_df=biome_dfs["Tropical"]

In [70]:
threshold = test_df['GC'].median()  # You can choose any threshold based on your needs

df_upper = test_df[test_df['GC'] >= threshold]
df_lower = test_df[test_df['GC'] < threshold]

In [71]:
df_upper.columns

Index(['PID', 'Raos_Q', 'Functional_Evenness', 'Mean Pairwise D',
       'CHELSA_BIO_Annual_Mean_Temperature', 'CHELSA_BIO_Annual_Precipitation',
       'CHELSA_BIO_Precipitation_Seasonality',
       'CrowtherLab_SoilMoisture_intraAnnualSD_downsampled10km',
       'SG_SOC_Content_015cm', 'EsaCci_BurntAreasProbability',
       'EarthEnvTopoMed_Elevation', 'EarthEnvTopoMed_Slope',
       'SG_Depth_to_bedrock', 'EarthEnvTopoMed_Northness', 'BHAGE', 'managed',
       'ownership', 'biome', 'transformed npp', 'GC', 'lat', 'lon', 'DIA',
       'lon_bin', 'lat_bin', 'spatial_group', 'geometry', 'index_right',
       'ECO_NAME'],
      dtype='str')

In [72]:
def train_test_split(sub_df, random_key, test_size):
    sub_df.drop(columns=['BHAGE', 'managed',
        'ownership', 'biome', 'geometry', 'index_right', 'lat', 'lon', 'DIA',
        'lon_bin', 'lat_bin', 'GC'], inplace=True)

    sets=list(set(sub_df['ECO_NAME']))

    random.seed(random_key)
    test_set = random.sample(sets, int(test_size * len(sets)))
    test = sub_df[sub_df["ECO_NAME"].isin(test_set)]
    train = sub_df[~sub_df["ECO_NAME"].isin(test_set)]

    y=['transformed npp'] 

    X_train=train.drop(columns=y+['PID', 'spatial_group', 'ECO_NAME'])
    y_train = train['transformed npp'].values

    X_test= test.drop(columns=y+['PID', 'spatial_group', 'ECO_NAME'])
    y_test = test['transformed npp'].values

    return X_train, y_train, X_test, y_test

In [73]:
X_train, y_train, X_test, y_test = train_test_split(df_upper, 21, 0.3)


In [ ]:
fd_reg = RandomForestRegressor(random_state=RANDOM_KEY, n_jobs=2)
fd_reg.fit(X_train, y_train)

fimp_df, metric_df = evaluate_rf(X_test, y_test, fd_reg,
                feature_names=X_train.columns, 
                div_type= div_type, 
                biome=biome_name,
                color=color)

fimp_df["biome"] = biome_name
metric_df["biome"] = biome_name

# Feature_importance_df = pd.concat([Feature_importance_df, fimp_df], ignore_index=True)
# Performance_df = pd.concat([Performance_df, metric_df], ignore_index=True)


In [67]:

import pandas as pd
import numpy as np

from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder

import matplotlib.pyplot as plt
import os
from pathlib import Path
import shutil

def evaluate_rf(X_test, y_test, regr, feature_names: list, importance: bool, div_type: str, biome: str = None, color=None):

    y_pred = regr.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred, multioutput='raw_values')

    r2 = r2_score(y_test, y_pred, multioutput='raw_values')

    metric_df = pd.DataFrame({
    "metric": ["r2", "mae"],
    "npp": [r2, mae],
})

    r2_str = ", ".join(f"{x:.3f}" for x in np.atleast_1d(r2))
    mae_str = ", ".join(f"{x:.3f}" for x in np.atleast_1d(mae))

    if importance: 
        importances = regr.feature_importances_

        if div_type == "f":
            f_list= ['Raos_Q', 'Functional_Evenness', "Mean Pairwise D"]
            a= 'Functional Diversity'

        elif div_type == "comb":
            f_list= ['Raos_Q', 'Functional_Evenness', "Mean Pairwise D"]
            a= 'Functional Diversity'

        else:
            f_list=['Species Richness', 'Shannon Diversity', "Simpson's Index"]
            a='Species Diversity'

        feature_imp_df = pd.DataFrame({'Feature': feature_names, 'Gini Importance': importances})
        
        
        feature_imp_df = feature_imp_df.sort_values('Gini Importance', ascending=False)

    return feature_imp_df, metric_df

